In [0]:
from pyspark.sql.functions import * 

df = spark.table("data_engineering.bronze_layer.bronze_sales")

df = df.withColumn("sales", expr("try_cast(sales as double)"))

df= df.withColumn("sales", coalesce(col("sales"), lit(0)))

df = df.fillna({"city": "unknown"})

df = df.withColumn("city", lower(col("city")))

print(f"record count before dropduplicate:{df.count()}")

#Removing duplicates
df = df.dropDuplicates()

print(f"record count after dropduplicate:{df.count()}")

df = df.withColumn("ingestion_date", current_timestamp())

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("data_engineering.silver_layer.silver_sales")